****
IMPORT
****

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

****
TRAIN / TEST
****

In [2]:
# 1. Chargement du dataset de training choisi
df_ml = pd.read_parquet("../data/dataframe_training.parquet")

# 2. Séparation de la cible (y) et des caractéristiques (X)
# On supprime explicitement la cible et l'identifiant pour la matrice X
X = df_ml.drop(columns=['a_quitte_l_entreprise', 'id_user'], errors='ignore')
y = df_ml['a_quitte_l_entreprise']

# 3. Création des jeux d'apprentissage et de test (Stratified pour équilibrer le turnover)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"📦 Jeux d'apprentissage et de test créés avec succès !")
print(f"Train set (X_train) : {X_train.shape[0]} lignes | Test set (X_test) : {X_test.shape[0]} lignes")

📦 Jeux d'apprentissage et de test créés avec succès !
Train set (X_train) : 1176 lignes | Test set (X_test) : 294 lignes


****
MODELES
****

In [4]:
# 1. Modèle Baseline / Dummy (Prédit toujours la classe la plus fréquente)
model_dummy = DummyClassifier(strategy="most_frequent")
model_dummy.fit(X_train, y_train)

# 2. Modèle Linéaire (Régression Logistique)
# max_iter augmenté pour garantir la convergence mathématique
model_linear = LogisticRegression(max_iter=1000, random_state=42)
model_linear.fit(X_train, y_train)

# 3. Modèle Non-Linéaire (Random Forest - Optimal pour le Label Encoding)
model_nonlinear = RandomForestClassifier(n_estimators=100, random_state=42)
model_nonlinear.fit(X_train, y_train)

print("🎯 Les 3 modèles ont été entraînés avec succès sur le jeu d'apprentissage !")

🎯 Les 3 modèles ont été entraînés avec succès sur le jeu d'apprentissage !


c:\Users\Nérion\Documents\Code\5_OpenClassRooms\3_automatically_classify_information\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


`DETAIL`
- Dummy => Établit la baseline la plus logique face à un dataset déséquilibré. Le modèle prédit aveuglément la situation majoritaire, ici l'employé qui reste.
- Linear (max_iter=1000) => Augmenté par rapport à la valeur par défaut (100) pour garantir la convergence mathématique de l'algorithme. Les données contenant plusieurs features créées et des échelles différentes, cela donne au solveur linéaire le temps de trouver la frontière de décision optimale sans planter.
- NonLinear (n_estimators=100) => Paramètre standard et robuste qui instancie un ensemble de 100 arbres de décision indépendants. Ce nombre offre un excellent compromis entre puissance prédictive et temps de calcul, tout en réduisant la variance globale par rapport à un arbre de décision unique.
- random_state=42 => Fixe la graine aléatoire pour garantir la reproductibilité stricte des résultats.

****
METRIQUES
****

In [6]:
# Fonction d'extraction automatique des métriques
def extraire_metriques(model, X, y, nom_jeu):
    y_pred = model.predict(X)
    return {
        f"Accuracy ({nom_jeu})": accuracy_score(y, y_pred),
        f"Precision ({nom_jeu})": precision_score(y, y_pred, zero_division=0),
        f"Recall ({nom_jeu})": recall_score(y, y_pred, zero_division=0),
        f"F1-Score ({nom_jeu})": f1_score(y, y_pred, zero_division=0)
    }

# Compilation des résultats pour chaque modèle
dict_resultats = {}

for nom, model in [("Modèle Dummy", model_dummy), ("Modèle Linéaire (LogReg)", model_linear), ("Modèle Non-Linéaire (RF)", model_nonlinear)]:
    # Métriques d'apprentissage
    metriques_train = extraire_metriques(model, X_train, y_train, "Train")
    # Métriques de validation (Test)
    metriques_test = extraire_metriques(model, X_test, y_test, "Test")
    
    # Fusion des dictionnaires pour ce modèle
    dict_resultats[nom] = {**metriques_train, **metriques_test}

# Conversion en DataFrame pour affichage propre sous forme de tableau
df_metriques = pd.DataFrame(dict_resultats).T

# Réorganisation esthétique des colonnes (Train à côté de Test)
colonnes_ordonnees = [
    "Accuracy (Train)", "Accuracy (Test)",
    "Precision (Train)", "Precision (Test)",
    "Recall (Train)", "Recall (Test)",
    "F1-Score (Train)", "F1-Score (Test)"
]

# Affichage du tableau final arrondi à 3 décimales
display(df_metriques[colonnes_ordonnees].round(3))

,Accuracy (Train),Accuracy (Test),Precision (Train),Precision (Test),Recall (Train),Recall (Test),F1-Score (Train),F1-Score (Test)
Modèle Dummy,0.838,0.840,0.000,0.000,0.000,0.000,0.000,0.000
Modèle Linéaire (LogReg),0.860,0.844,0.745,0.533,0.200,0.170,0.315,0.258
Modèle Non-Linéaire (RF),0.999,0.827,1.000,0.375,0.995,0.128,0.997,0.190


`OBSERVATION:`
- Dummy => Ce modèle "naïf" se contente de prédire la classe majoritaire (l'employé reste) car le dataset est déséquilibré. Il prouve que l'Accuracy est une métrique trompeuse ici : un modèle peut sembler performant tout en étant totalement incapable de détecter la moindre démission
- Linear => Sa précision indique qu'il est capable de prédir plus d'un départ sur 2. Moins sensible au surapprentissage que le Random Forest, il reste cependant limité par le déséquilibre des classes. Son Recall de 0.170 montre qu'il ne parvient pas à intercepter efficacement les signaux faibles de turnover.
- NonLinear => Des scores quasi-parfaits sur le jeu d'entraînement qui s'effondrent sur le jeu de test. C'est un cas d'overfitting.

``Ces résultats indique clairement que le dataset est trop déséquilibré et nécessite un rééquilibrage ou l'adaptation de paramètre des modèles.``